# 02 - Cotizaciones y datos de mercado

Recorrido por los endpoints de market data de la API v1.

## Que se ve en este notebook

- Cotizacion de punta de un activo.
- Profundidad del libro de ordenes.
- Tipos de activo y paneles, para usar el screener.
- Screener de un panel.
- Cadena de opciones de un subyacente.
- Serie historica OHLCV.

Todo es de solo lectura: no se envia ninguna orden.


## 1. Credenciales

In [ ]:
import os

# completar: exportar las credenciales en el entorno antes de ejecutar.
USERNAME = os.environ.get("IOL_USERNAME", "")  # completar
PASSWORD = os.environ.get("IOL_PASSWORD", "")  # completar

if not USERNAME or not PASSWORD:
    raise SystemExit("Faltan IOL_USERNAME e IOL_PASSWORD en el entorno")


## 2. Imports

In [ ]:
from pyIol2 import IOLAPIError, IOLClient, Markets, SettlementTerms

SYMBOL = "GGAL"  # cambiar por el titulo que se quiera mirar


## 3. Cotizacion de mercado

`get_market_quote(symbol, market, term)` devuelve `MarketQuote`: `prices`, `change` y `volume`.

In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    cotizacion = client.get_market_quote(
        SYMBOL,
        market=Markets.BCBA,
        term=SettlementTerms.T0,
    )
    print(cotizacion.symbol, cotizacion.market, cotizacion.settlement_term)
    print("ultimo:", cotizacion.prices.current, cotizacion.currency)
    print("apertura:", cotizacion.prices.open, "cierre anterior:", cotizacion.prices.previous_close)
    print("variacion:", cotizacion.change.amount, cotizacion.change.percent, "%")
    print("volumen:", cotizacion.volume.shares, "nominales por", cotizacion.volume.amount)
    print("actualizado:", cotizacion.last_updated)


## 4. Profundidad del libro

`get_market_depth()` devuelve las cinco mejores puntas de compra (`bids`) y de venta (`asks`).

In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    libro = client.get_market_depth(SYMBOL)
    print("compras:")
    for nivel in libro.bids:
        print(" ", nivel.position, nivel.price, nivel.size)
    print("ventas:")
    for nivel in libro.asks:
        print(" ", nivel.position, nivel.price, nivel.size)


## 5. Screener

Primero se descubren los ids: `get_asset_types()` y `get_asset_panels()`.
Los ids son datos de la API, no alias fijos como `"acciones"` o `"merval"`.


In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    tipos = client.get_asset_types()
    for tipo in tipos:
        print(tipo.id, tipo.name)


In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    paneles = client.get_asset_panels(tipos[0].id) if tipos else []
    for panel in paneles:
        print(panel.id, panel.name)


In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    if tipos and paneles:
        filas = client.get_screener_quotes(tipos[0].id, paneles[0].id)
        print("instrumentos:", len(filas))
        for fila in filas[:5]:
            print(
                fila.instrument.symbol,
                fila.quotes.last,
                fila.quotes.change_percent,
                "| compra",
                fila.book.bid.price,
                "venta",
                fila.book.ask.price,
            )
    else:
        print("no hay tipos o paneles para mostrar")


## 6. Cadena de opciones

`get_option_chain(symbol, market)` devuelve `OptionQuote` con `option.strike_price`, `option.type` y la punta del libro.

In [ ]:
with IOLClient(USERNAME, PASSWORD) as client:
    opciones = client.get_option_chain(SYMBOL, market=Markets.BCBA)
    print("opciones:", len(opciones))
    for opcion in opciones[:10]:
        print(
            opcion.symbol,
            opcion.option.type,
            opcion.option.strike_price,
            opcion.quotes.last,
            opcion.book.bid.price,
            opcion.book.ask.price,
        )


## 7. Serie historica

Las fechas van como `YYYY-MM-DD HH:MM:SS`.

In [ ]:
from datetime import datetime, timedelta

hasta = datetime.now()
desde = hasta - timedelta(days=30)

with IOLClient(USERNAME, PASSWORD) as client:
    serie = client.get_historical_bars(
        SYMBOL,
        desde.strftime("%Y-%m-%d 00:00:00"),
        hasta.strftime("%Y-%m-%d 23:59:59"),
    )
    print(serie.symbol, serie.market, serie.total_bars, "velas")
    for vela in serie.bars[-5:]:
        print(vela.datetime, vela.open, vela.high, vela.low, vela.close, vela.volume)


## 8. Errores

Un simbolo inexistente llega como `IOLNotFoundError`; el `trace_id` es el dato que pide el soporte de IOL.

In [ ]:
try:
    with IOLClient(USERNAME, PASSWORD) as client:
        client.get_market_quote("SIMBOLO-INEXISTENTE")
except IOLAPIError as exc:
    print(type(exc).__name__, exc.status_code, exc.code)
    print("trace_id:", exc.trace_id)
